# Notebook to try to improve intial QA generation prompt

In [ ]:
import logging
import textwrap
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
from tqdm import tqdm

from Wallis.RAGs.RAGv3 import VectorBase

logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)
from langchain_ollama import ChatOllama

In [ ]:
def print_wrap(text, width=80):
    wrapped = textwrap.fill(text, width=width)
    print(wrapped)

In [ ]:
chunk_size = 6000
chunk_overlap = 1
min_groundness = 3
min_relevance = 3
model_name = "llama3.1:8b"
model_name_70 = "llama3.3"

In [ ]:
vectorebase = VectorBase(
    directory="../../transformed_documents",
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

In [ ]:
chunks = vectorebase.chunks

In [ ]:
np.median([len(chunk.page_content.split()) for chunk in chunks])

In [ ]:
llm = ChatOllama(model=model_name, num_predict=1000, temperature=0)
llm_70 = ChatOllama(model=model_name_70, num_predict=600, temperature=0)

### Try MCQ

In [ ]:
# MCQ prompting
QA_generation_prompt = """
Your task is to write 10 factoid multiple choices questions and their answer given a context. Only one proposed choice per question must be correct.
Your factoid questions should be about a specific, concise piece of factual information from the context.
Your factoid questions must not mention something like "according to the passage" or "context" or "according to this article" but on the contrary directly cite what is the context.
Write the questions and the answers in french.
A bad example is "What is the name of the law ?" because we don't know which law you are talking about.
Detail your answer, giving an explanation and always citing from which article or text of law does the answer come from. But keep it concise.
Someone should be able to answer the question without the context.
Provide your answer as follows, please do not add any spaces character and absolutely respect this format, do not include anything else in your answer:
Output:::
Question 1: (your factoid question)
Choices:
A: (choice A an answer for the factoid question. Source: Law n°... , Article...))
B: (choice B)
C: (choice C)
D: (choice D)
Réponse 1: A, B, C or D (the letter of the correct answer)

Question 2: (your factoid question)
Choices:
A: (choice A an answer for the factoid question. Source: Law n°... , Article...))
B: (choice B)
C: (choice C)
D: (choice D)
Réponse 2: (the letter of the correct answer)

...
Question 10: (your factoid question)
Choices:
A: (choice A an answer for the factoid question. Source: Law n°... , Article...))
B: (choice B)
C: (choice C)
D: (choice D)
Réponse 10: (the letter of the correct answer)

Now here is the context.

Context: {context}\n
Output:::"""

In [ ]:
QA_generation_prompt = """
Your task is to write 10 non trivial factoid multiple choices questions and their answer given a context. Only one proposed choice per question must be correct.
Your factoid questions should be about a specific, concise piece of factual information from the context.
Your factoid questions must not mention something like "according to the passage" or "context" or "according to this article" but on the contrary directly cite what is the context.
Write the questions and the answers in french.
A bad example is "What is the name of the law ?" because we don't know which law you are talking about.
Detail your answer, giving an explanation and always citing from which article or text of law does the answer come from. But keep it concise.
Someone should be able to answer the question without the context. 
Ypu can create false answers by giving a good answer but citing the wrong article or law.
Provide your answer as follows, please do not add any spaces character and absolutely respect this format, do not include anything else in your answer:
Output:::
Question 1: (your factoid question)
Choices:
A: (choice A an answer for the factoid question. Source: Law n°... , Article...))
B: (choice B. (Source: Law n°... , Article...))
C: (choice C. (Source: Law n°... , Article...))
D: (choice D. (Source: Law n°... , Article...))

Question 2: (your factoid question)
Choices:
A: (choice A an answer for the factoid question. (Source: Law n°... , Article...))
B: (choice B. (Source: Law n°... , Article...))
C: (choice C. (Source: Law n°... , Article...))
D: (choice D. (Source: Law n°... , Article...))

...
Question 10: (your factoid question)
Choices:
A: (choice A an answer for the factoid question. Source: Law n°... , Article...))
B: (choice B)
C: (choice C)
D: (choice D)

Now here is the context.

Context: {context}\n
Output:::"""

In [ ]:
old_QA_generation_prompt = """
Your task is to write 10 factoid questions and their answer given a context.
Your factoid questions should be about a specific, concise piece of factual information from the context.
Your factoid questions should be formulated in the same style as questions users could ask in a search engine, maximum 9 words.
This means that your factoid questions MUST NOT mention something like "according to the passage" or "context" but on the contrary directly cite what is the context.
Write the questions and the answers in french.
For example, "What is the name of the law ?" is a bad question  and must be avoided because it cannot be understood without the context.
Instead, "What is the name of the law that prohibits smoking in public places?" is a good question.
Also, "How many members are in the commission?" is a bad question because we don't know which commission you are talking about.
Provide your answer as follows, please do not add any spaces character and absolutely respect this format, do not include anything else in your answer:
Output:::
Question 1: (your factoid question)
Réponse 1: (your answer to the factoid question)
Question 2: (your factoid question)
Réponse 2: (your answer to the factoid question)
...
Question 10: (your factoid question)
Réponse 10: (your answer to the factoid question)

Now here is the context.

Context: {context}\n
Output:::"""

In [ ]:
random_chunk = np.random.choice(chunks)
context_example = random_chunk.page_content
prompt = QA_generation_prompt.format(context=context_example)
%time generated_questions=llm.invoke(prompt)
# old_generated_questions=old_QA_generation_prompt.format(context=context_example)
# old_QA_generation_questions=llm.invoke(old_generated_questions)
print(random_chunk.metadata["title"])
print("new:")
generated_questions.content.split("\n")


# print_wrap(generated_questions70.content)

answering seems to easy

## Normal questions (open questions)

In [ ]:
def parse_questions_answers(generated_questions):
    """Parse the generated questions and answers."""
    questions = generated_questions.content.split("Question")
    QA_pairs = []
    for question in questions:
        if "Réponse" not in question:
            continue
        qu, ans = question.split("Réponse")
        qu = qu[3:].strip(":").strip()
        ans = ans[3:].strip(":").strip()
        QA_pairs.append({"question": qu, "answer": ans})
    return QA_pairs

In [ ]:
parse_questions_answers(generated_questions)

In [ ]:
data = generated_questions_70.content.split("\n")
questions_and_answers = []

# Iterate through the list in steps of 2, extracting question and answer pairs
for i in range(0, len(data), 3):  # Step by 3 to skip empty strings
    print(data[i])

    question = data[i].replace("Question ", "")[2:].strip()
    answer = data[i + 1].replace("Réponse ", "")[2:].strip()
    questions_and_answers.append({"question": question, "answer": answer})

# Print the result
print(questions_and_answers)

In [ ]:
def parse_questions_answers(generated_questions):
    data = generated_questions.content.split("\n")
    questions_and_answers = []

    # Iterate through the list in steps of 2, extracting question and answer pairs
    for i in range(0, len(data), 3):  # Step by 3 to skip empty strings
        try:
            question = data[i].replace("Question ", "")[2:].strip()
            answer = data[i + 1].replace("Réponse ", "")[2:].strip()
            questions_and_answers.append({"question": question, "answer": answer})
        except:
            print("Error in parsing")
            print(data[i : i + 3])
            print("wholde data:")
            print(data)

    return questions_and_answers

In [ ]:
def create_qa(chunk, llm):
    """Generate question and answer pairs for a given chunk of text."""
    context = chunk.page_content
    prompt = QA_generation_prompt.format(context=context)
    generated_questions = llm.invoke(prompt)
    generated_questions = parse_questions_answers(generated_questions)
    dico_res = {
        "title": chunk.metadata["title"],
        "generated_questions": generated_questions,
        "context": context,
    }

    return dico_res

In [ ]:
chunks = chunks[:50]

In [ ]:
def process_chunks_parallel(chunks, llm=llm, max_workers=4):
    """Process chunks in parallel using ThreadPoolExecutor."""
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(create_qa, chunk, llm): chunk for chunk in chunks}

        for future in tqdm(
            as_completed(futures), total=len(chunks), desc="Processing Chunks"
        ):
            results.append(future.result())

    formatted_results = []
    for result in results:
        for qa in result["generated_questions"]:
            formatted_results.append(
                {
                    "title": result["title"],
                    "question": qa["question"],
                    "answer": qa["answer"],
                    "context": result["context"],
                }
            )

    return formatted_results

In [ ]:
results = process_chunks_parallel(chunks[45:], max_workers=8)

### Create impossible questions

In [ ]:
QA_generation_prompt_impossible = """
Your task is to write 2 factoid questions that are impossible to answer given the context.
Your factoid questions should be formulated in the same style as questions users could ask in a search engine, maximum 9 words.
Your factoid questions MUST directly say what the context is about but it MUST be impossible to answer the question with the given context.
For example: "What is the date of th national holiday?" is good if the date of the national holiday is not in the context.
Write the questions in french.
Provide your answer as follows, please do not add any spaces character and absolutely respect this format, do not include anything else in your answer:
Output:::
Question 1: (your factoid question)
Question 2: (your factoid question)

Now here is the context.

Context: {context}\n
Remember that the questions must be impossible to answer with the given context.
Output:::
"""

In [ ]:
chunk = np.random.choice(chunks)
context_example = chunk.page_content
title = chunk.metadata["title"]
print(f"title: {title}")
prompt = QA_generation_prompt_impossible.format(context=context_example)
generated_questions = llm.invoke(prompt)
print_wrap(generated_questions.content)

In [ ]:
generated_questions.content.split("\n")